#review por localidade#

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType

review_df = spark.table("workspace.yelp_ing.silver_review").limit(100)
business_df = spark.table("workspace.yelp_ing.silver_business").limit(100)

avg_stars_df = review_df.groupBy("business_id").agg(F.avg("stars").alias("average_stars"))

def is_closed_on_review_day(hours, review_date):
    import json
    from datetime import datetime
    if not hours:
        return False
    try:
        hours_dict = json.loads(hours)
    except Exception:
        return False
    weekday = datetime.strptime(review_date, "%Y-%m-%d").strftime("%A").lower()
    day_hours = hours_dict.get(weekday)
    if not day_hours or day_hours.lower() == "closed":
        return True
    return False

is_closed_udf = F.udf(is_closed_on_review_day, BooleanType())

joined_df = review_df.join(
    business_df,
    on="business_id",
    how="inner"
).join(
    avg_stars_df,
    on="business_id",
    how="inner"
).select(
    review_df.business_id,
    review_df.date,
    review_df.review_id,
    review_df.stars,
    review_df.text,
    review_df.user_id,
    business_df.food_category,
    business_df.city,
    business_df.state,
    business_df.hours,
    avg_stars_df.average_stars,
    business_df.review_count.alias("review_count_silver_business")
).withColumn(
    "score_yelp",
    F.col("average_stars") * F.col("review_count_silver_business")
).withColumn(
    "fraud_suspicious",
    is_closed_udf(F.col("hours"), F.col("date"))
)

display(joined_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType

def is_closed_on_review_day(hours, review_date):
    import json
    from datetime import datetime
    if not hours:
        return False
    try:
        hours_dict = json.loads(hours)
    except Exception:
        return False
    weekday = datetime.strptime(review_date, "%Y-%m-%d").strftime("%A").lower()
    day_hours = hours_dict.get(weekday)
    if not day_hours or day_hours.lower() == "closed":
        return True
    return False

is_closed_udf = F.udf(is_closed_on_review_day, BooleanType())

review_df = spark.table("workspace.yelp_ing.silver_review").limit(100)
business_df = spark.table("workspace.yelp_ing.silver_business").limit(100)

avg_stars_df = review_df.groupBy("business_id").agg(F.avg("stars").alias("average_stars"))

joined_df = review_df.join(
    business_df,
    on="business_id",
    how="inner"
).join(
    avg_stars_df,
    on="business_id",
    how="inner"
).select(
    review_df.business_id,
    review_df.date,
    review_df.review_id,
    review_df.stars,
    review_df.text,
    review_df.user_id,
    business_df.food_category,
    business_df.city,
    business_df.state,
    business_df.hours,
    avg_stars_df.average_stars,
    business_df.review_count.alias("review_count_silver_business")
).withColumn(
    "score_yelp",
    F.col("average_stars") * F.col("review_count_silver_business")
).withColumn(
    "fraud_suspicious",
    is_closed_udf(F.col("hours"), F.col("date"))
)

display(joined_df)